
# STEP 35A — Table 3 Verifier
## BrainFMOps-Analyze Publication Audit

Notebook นี้ตรวจความน่าเชื่อถือของ **Table 3: Labelled vs Unlabelled Cohort Comparison** แบบอัตโนมัติ

ตรวจ 6 ด้าน:
1. Subject merge integrity
2. Age verification
3. Sex verification
4. Readiness-warning audit
5. Slice-count availability
6. Statistical re-computation

Input หลัก:
```text
<repository-root>\
├── 34_Table_Generator_Input\
│   ├── evaluation_summary_with_labels.csv
│   └── oasis_cross-sectional*.xlsx
└── 35_Table3_Output\
    └── Table3_Labelled_vs_Unlabelled.xlsx
```

Output:
```text
<repository-root>\35A_Table3_Verifier_Output
```

> ใช้งาน: `Kernel → Restart & Run All`


In [ ]:

from pathlib import Path
import json, re, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")

ROOTS = [
    Path.cwd().resolve(),
    Path.cwd().resolve(),
    Path.cwd(),
]
ROOT = next((p for p in ROOTS if p.exists()), Path.cwd())

INPUT_DIR = ROOT / "34_Table_Generator_Input"
TABLE3_DIR = ROOT / "35_Table3_Output"
OUTPUT_DIR = ROOT / "35A_Table3_Verifier_Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRED_FILE = INPUT_DIR / "evaluation_summary_with_labels.csv"
CLINICAL_FILES = sorted(
    list(INPUT_DIR.glob("oasis-cross-sectional*.xlsx"))
    + list(INPUT_DIR.glob("oasis_cross-sectional*.xlsx"))
    + list(INPUT_DIR.glob("oasis-cross-sectional*.csv"))
    + list(INPUT_DIR.glob("oasis_cross-sectional*.csv"))
)
TABLE3_FILE = TABLE3_DIR / "Table3_Labelled_vs_Unlabelled.xlsx"

if not PRED_FILE.exists():
    raise FileNotFoundError(PRED_FILE)
if not CLINICAL_FILES:
    raise FileNotFoundError("No OASIS cross-sectional clinical file found")
if not TABLE3_FILE.exists():
    raise FileNotFoundError(TABLE3_FILE)

CLINICAL_FILE = CLINICAL_FILES[0]

print("Prediction:", PRED_FILE)
print("Clinical  :", CLINICAL_FILE)
print("Table 3   :", TABLE3_FILE)
print("Output    :", OUTPUT_DIR)


In [ ]:

def load_table(path):
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    return pd.read_excel(path)

def first_existing(columns, candidates):
    lookup = {str(c).strip().lower(): c for c in columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    return None

def normalize_subject_id(v):
    if pd.isna(v): return np.nan
    t = str(v).strip().upper().replace("-", "_").replace(" ", "_")
    t = re.sub(r"_+", "_", t)
    m = re.search(r"(OAS1_\d{4}_MR\d+)", t)
    return m.group(1) if m else t

def normalize_label(v):
    if pd.isna(v): return np.nan
    t = str(v).strip().upper()
    if t in {"CN","0","CONTROL","NORMAL","COGNITIVELY NORMAL"}:
        return "CN"
    if t in {"AD","1","ALZHEIMER","ALZHEIMER'S DISEASE","DEMENTIA","DEMENTED"}:
        return "AD"
    return np.nan

def sex_norm(v):
    if pd.isna(v): return np.nan
    t = str(v).strip().upper()
    return {"F":"F","FEMALE":"F","M":"M","MALE":"M"}.get(t, np.nan)

def welch_p(a,b):
    a = pd.to_numeric(a,errors="coerce").dropna()
    b = pd.to_numeric(b,errors="coerce").dropna()
    if len(a)<2 or len(b)<2: return np.nan
    return float(stats.ttest_ind(a,b,equal_var=False).pvalue)

def mann_p(a,b):
    a = pd.to_numeric(a,errors="coerce").dropna()
    b = pd.to_numeric(b,errors="coerce").dropna()
    if len(a)==0 or len(b)==0: return np.nan
    return float(stats.mannwhitneyu(a,b,alternative="two-sided").pvalue)

def chi_or_fisher(table):
    arr = np.asarray(table,dtype=int)
    if arr.shape != (2,2) or arr.sum()==0: return np.nan,"NA"
    chi2,p,dof,expected = stats.chi2_contingency(arr)
    if (expected < 5).any():
        _,p = stats.fisher_exact(arr)
        return float(p),"Fisher exact"
    return float(p),"Chi-square"

def fmt_p(p):
    if p is None or not np.isfinite(p): return "—"
    return "<0.001" if p < 0.001 else f"{p:.3f}"

def fmt_mean_sd(s,digits=1):
    x = pd.to_numeric(s,errors="coerce").dropna()
    if len(x)==0: return "NA"
    return f"{x.mean():.{digits}f} ± {x.std(ddof=1):.{digits}f}"

def fmt_n_pct(n,total):
    if total==0: return "0 (0.0%)"
    return f"{int(n)} ({100*n/total:.1f}%)"


In [ ]:

pred = load_table(PRED_FILE)
clinical = load_table(CLINICAL_FILE)
table3 = pd.read_excel(TABLE3_FILE, sheet_name="Table 3", skiprows=2)

print("Prediction shape:", pred.shape)
print("Clinical shape  :", clinical.shape)
print("Table3 shape    :", table3.shape)


In [ ]:

# Detect key columns
id_col = first_existing(pred.columns, ["case_id","subject_key","subject_id","id"])
label_col = first_existing(pred.columns, ["ground_truth","ground_truth_original","clinical_label"])
clinical_id_col = first_existing(clinical.columns, ["ID","subject_id","subject","case_id"])

if id_col is None or label_col is None or clinical_id_col is None:
    raise KeyError("Required ID/label columns could not be detected")

pred["_subject_id"] = pred[id_col].map(normalize_subject_id)
pred["_label"] = pred[label_col].map(normalize_label)
pred["_cohort"] = np.where(pred["_label"].notna(),"Labelled","Unlabelled")
pred = pred.drop_duplicates("_subject_id",keep="first").copy()

clinical["_subject_id"] = clinical[clinical_id_col].map(normalize_subject_id)
clinical = clinical.drop_duplicates("_subject_id",keep="first").copy()

merged = pred.merge(
    clinical,
    on="_subject_id",
    how="left",
    suffixes=("_pred","_clinical"),
    indicator=True
)

labelled = merged[merged["_cohort"]=="Labelled"].copy()
unlabelled = merged[merged["_cohort"]=="Unlabelled"].copy()

print("Processed :", len(merged))
print("Labelled  :", len(labelled))
print("Unlabelled:", len(unlabelled))
print("Matched clinical:", int((merged["_merge"]=="both").sum()))


In [ ]:

# Module 1 — Merge integrity
dup_pred = int(pred["_subject_id"].duplicated().sum())
dup_clin = int(clinical["_subject_id"].duplicated().sum())
unmatched_n = int((merged["_merge"]!="both").sum())

merge_check = pd.DataFrame({
    "Check":[
        "Processed examinations",
        "Labelled examinations",
        "Unlabelled examinations",
        "Clinical matches",
        "Unmatched clinical records",
        "Duplicate prediction IDs",
        "Duplicate clinical IDs"
    ],
    "Value":[
        len(merged), len(labelled), len(unlabelled),
        int((merged["_merge"]=="both").sum()),
        unmatched_n, dup_pred, dup_clin
    ],
    "Expected":[401,212,189,401,0,0,0]
})
merge_check["Pass"] = merge_check["Value"] == merge_check["Expected"]
display(merge_check)


In [ ]:

# Module 2 — Age verification
age_col = first_existing(merged.columns, ["Age","age_years"])

if age_col is None:
    raise KeyError("Age column not found")

L_age = pd.to_numeric(labelled[age_col],errors="coerce")
U_age = pd.to_numeric(unlabelled[age_col],errors="coerce")

age_stats = pd.DataFrame({
    "Cohort":["Labelled","Unlabelled"],
    "n":[L_age.notna().sum(),U_age.notna().sum()],
    "Missing":[L_age.isna().sum(),U_age.isna().sum()],
    "Mean":[L_age.mean(),U_age.mean()],
    "SD":[L_age.std(ddof=1),U_age.std(ddof=1)],
    "Median":[L_age.median(),U_age.median()],
    "Min":[L_age.min(),U_age.min()],
    "Max":[L_age.max(),U_age.max()],
})
display(age_stats)

p_age_welch = welch_p(L_age,U_age)
p_age_mw = mann_p(L_age,U_age)

print("Welch p:", p_age_welch)
print("Mann-Whitney p:", p_age_mw)


In [ ]:

# Age plausibility analysis
# We do not label young controls as errors. Instead, we quantify how much of the
# unlabelled cohort is young and whether the large mean difference is data-driven.
thresholds = [30,40,50,60]

age_composition = []
for t in thresholds:
    age_composition.append({
        "Age threshold": f"< {t}",
        "Labelled n": int((L_age < t).sum()),
        "Labelled %": round(100*(L_age < t).mean(),1),
        "Unlabelled n": int((U_age < t).sum()),
        "Unlabelled %": round(100*(U_age < t).mean(),1),
    })

age_composition = pd.DataFrame(age_composition)
display(age_composition)


In [ ]:

# Age plots
plt.figure(figsize=(7,5))
plt.hist(L_age.dropna(), bins=20, alpha=0.6, label="Labelled")
plt.hist(U_age.dropna(), bins=20, alpha=0.6, label="Unlabelled")
plt.xlabel("Age (years)")
plt.ylabel("Frequency")
plt.title("Age Distribution: Labelled vs Unlabelled")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"Age_Distribution.png", dpi=300)
plt.close()

plt.figure(figsize=(6,5))
plt.boxplot([L_age.dropna(),U_age.dropna()], labels=["Labelled","Unlabelled"])
plt.ylabel("Age (years)")
plt.title("Age Comparison")
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"Age_Boxplot.png", dpi=300)
plt.close()

print("Age plots saved")


In [ ]:

# Module 3 — Sex verification
sex_col = first_existing(merged.columns, ["M/F","sex","gender"])

if sex_col is not None:
    L_sex = labelled[sex_col].map(sex_norm)
    U_sex = unlabelled[sex_col].map(sex_norm)

    lf,lm = int((L_sex=="F").sum()),int((L_sex=="M").sum())
    uf,um = int((U_sex=="F").sum()),int((U_sex=="M").sum())
    p_sex, sex_test = chi_or_fisher([[lf,lm],[uf,um]])

    sex_check = pd.DataFrame({
        "Cohort":["Labelled","Unlabelled"],
        "Female":[lf,uf],
        "Male":[lm,um],
        "Female %":[100*lf/len(labelled),100*uf/len(unlabelled)],
        "Male %":[100*lm/len(labelled),100*um/len(unlabelled)],
    })
    display(sex_check)
    print("Sex test:", sex_test, "| p =", p_sex)
else:
    p_sex = np.nan
    sex_test = "NA"
    print("Sex column not found")


In [ ]:

# Module 4 — Readiness-warning audit
status_col = first_existing(merged.columns, ["readiness_status","status"])
readiness_success_col = first_existing(merged.columns, ["readiness_success"])
failure_step_col = first_existing(merged.columns, ["failure_step"])
error_message_col = first_existing(merged.columns, ["error_message"])

def normalize_text(s):
    return s.fillna("").astype(str).str.strip()

warning_components = pd.DataFrame(index=merged.index)

if status_col is not None:
    warning_components["status_warn"] = normalize_text(merged[status_col]).str.upper().str.contains("WARN")
else:
    warning_components["status_warn"] = False

if readiness_success_col is not None:
    warning_components["readiness_false"] = normalize_text(
        merged[readiness_success_col]
    ).str.upper().isin(["FALSE","0"])
else:
    warning_components["readiness_false"] = False

if failure_step_col is not None:
    fs = normalize_text(merged[failure_step_col]).str.upper()
    warning_components["failure_step_nonempty"] = ~fs.isin([
        "","NONE","NAN","NULL","RESOLVED","NOT_APPLICABLE","N/A"
    ])
else:
    warning_components["failure_step_nonempty"] = False

if error_message_col is not None:
    em = normalize_text(merged[error_message_col]).str.upper()
    warning_components["error_message_nonempty"] = ~em.isin([
        "","NONE","NAN","NULL","RESOLVED","OK","SUCCESS","NOT_APPLICABLE","N/A"
    ])
else:
    warning_components["error_message_nonempty"] = False

merged["_warning_verified"] = warning_components.any(axis=1)

warning_audit_rows = []
for component in warning_components.columns:
    warning_audit_rows.append({
        "Component":component,
        "Labelled n":int(warning_components.loc[labelled.index,component].sum()),
        "Unlabelled n":int(warning_components.loc[unlabelled.index,component].sum())
    })

warning_audit = pd.DataFrame(warning_audit_rows)
display(warning_audit)

lw = int(merged.loc[labelled.index,"_warning_verified"].sum())
uw = int(merged.loc[unlabelled.index,"_warning_verified"].sum())

p_warn, warn_test = chi_or_fisher([
    [lw,len(labelled)-lw],
    [uw,len(unlabelled)-uw]
])

print("Verified warnings labelled  :", lw)
print("Verified warnings unlabelled:", uw)
print("Warning test:", warn_test, "| p =", p_warn)


In [ ]:

# Module 4B — Inspect source text values behind warning inflation
value_reports = []

for col_name, col in [
    ("status",status_col),
    ("readiness_success",readiness_success_col),
    ("failure_step",failure_step_col),
    ("error_message",error_message_col),
]:
    if col is None:
        continue
    counts = (
        merged[col]
        .fillna("<NA>")
        .astype(str)
        .value_counts(dropna=False)
        .head(20)
        .reset_index()
    )
    counts.columns = ["Value","Count"]
    counts.insert(0,"Field",col_name)
    value_reports.append(counts)

warning_value_report = (
    pd.concat(value_reports,ignore_index=True)
    if value_reports else pd.DataFrame()
)

display(warning_value_report.head(80))


In [ ]:

# Module 5 — Slice-count verification
SLICE_CANDIDATES = [
    "slice_count","n_slices","num_slices","number_of_slices",
    "slices_analysed","slices_analyzed","selected_slice_count",
    "usable_slice_count","analyzed_slice_count","analysed_slice_count"
]
slice_col = first_existing(merged.columns,SLICE_CANDIDATES)

slice_report = pd.DataFrame({
    "Item":["Explicit slice-count column found","Column name"],
    "Value":[slice_col is not None, slice_col if slice_col is not None else "Not available"]
})
display(slice_report)

if slice_col is None:
    print("Publication-safe decision: keep slice count as Not available or remove row.")
else:
    print("Slice count found:", slice_col)


In [ ]:

# Module 6 — Re-compute Table 3 statistics
recalc_rows = []

recalc_rows.append({
    "Characteristic":"Age (years), mean ± SD",
    "Labelled":fmt_mean_sd(L_age,1),
    "Unlabelled":fmt_mean_sd(U_age,1),
    "p-value":fmt_p(p_age_welch),
    "Test":"Welch t-test"
})

if sex_col is not None:
    recalc_rows.append({
        "Characteristic":"Sex, female / male, n (%)",
        "Labelled":f"{fmt_n_pct(lf,len(labelled))} / {fmt_n_pct(lm,len(labelled))}",
        "Unlabelled":f"{fmt_n_pct(uf,len(unlabelled))} / {fmt_n_pct(um,len(unlabelled))}",
        "p-value":fmt_p(p_sex),
        "Test":sex_test
    })

recalc_rows.append({
    "Characteristic":"Readiness-assessment warnings, n (%)",
    "Labelled":fmt_n_pct(lw,len(labelled)),
    "Unlabelled":fmt_n_pct(uw,len(unlabelled)),
    "p-value":fmt_p(p_warn),
    "Test":warn_test
})

recalc_table = pd.DataFrame(recalc_rows)
display(recalc_table)


In [ ]:

# Publication decision logic
issues = []

if not merge_check["Pass"].all():
    issues.append("Cohort/clinical merge audit failed.")

if U_age.notna().sum() < 0.9*len(unlabelled):
    issues.append("Age data are missing for more than 10% of the unlabelled cohort.")

if age_stats.loc[age_stats["Cohort"]=="Unlabelled","Mean"].iloc[0] < 40:
    issues.append(
        "Unlabelled cohort is substantially younger than the labelled cohort; "
        "verify this is a real repository composition effect and discuss selection bias."
    )

if lw/len(labelled) > 0.5 or uw/len(unlabelled) > 0.5:
    issues.append(
        "Readiness-warning prevalence exceeds 50%; verify whether warning logic "
        "captures benign/resolved pipeline states."
    )

if slice_col is None:
    issues.append(
        "No explicit slice-count variable is available. Do not substitute selected_volume_index "
        "or selected_volume_score."
    )

status = "READY WITH CAVEATS" if issues else "READY FOR PUBLICATION"

verification_summary = {
    "status":status,
    "processed":len(merged),
    "labelled":len(labelled),
    "unlabelled":len(unlabelled),
    "clinical_matches":int((merged["_merge"]=="both").sum()),
    "age_labelled_mean":float(L_age.mean()),
    "age_unlabelled_mean":float(U_age.mean()),
    "age_welch_p":float(p_age_welch),
    "sex_p":None if not np.isfinite(p_sex) else float(p_sex),
    "verified_warnings_labelled":lw,
    "verified_warnings_unlabelled":uw,
    "warning_p":None if not np.isfinite(p_warn) else float(p_warn),
    "slice_count_column":slice_col,
    "issues":issues
}

print("STATUS:",status)
for issue in issues:
    print("-",issue)


In [ ]:

# Export audit workbook
audit_xlsx = OUTPUT_DIR/"Table3_Verification_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx,engine="openpyxl") as writer:
    merge_check.to_excel(writer,sheet_name="Merge Check",index=False)
    age_stats.to_excel(writer,sheet_name="Age Statistics",index=False)
    age_composition.to_excel(writer,sheet_name="Age Composition",index=False)
    if sex_col is not None:
        sex_check.to_excel(writer,sheet_name="Sex Check",index=False)
    warning_audit.to_excel(writer,sheet_name="Warning Audit",index=False)
    warning_value_report.to_excel(writer,sheet_name="Warning Values",index=False)
    slice_report.to_excel(writer,sheet_name="Slice Check",index=False)
    recalc_table.to_excel(writer,sheet_name="Recomputed Table3",index=False)

print("Saved:",audit_xlsx)


In [ ]:

# Create verifier report DOCX
report_path = OUTPUT_DIR/"Table3_Verification_Report.docx"

try:
    from docx import Document
    from docx.shared import Pt

    doc = Document()
    h = doc.add_heading("STEP 35A — Table 3 Verification Report", level=1)
    doc.add_paragraph(f"Overall status: {status}")

    doc.add_heading("1. Cohort integrity",level=2)
    doc.add_paragraph(
        f"Processed={len(merged)}, Labelled={len(labelled)}, "
        f"Unlabelled={len(unlabelled)}, Clinical matches="
        f"{int((merged['_merge']=='both').sum())}."
    )

    doc.add_heading("2. Age verification",level=2)
    doc.add_paragraph(
        f"Labelled age: {fmt_mean_sd(L_age,1)} years; "
        f"Unlabelled age: {fmt_mean_sd(U_age,1)} years; "
        f"Welch p={fmt_p(p_age_welch)}."
    )

    doc.add_heading("3. Sex verification",level=2)
    if sex_col is not None:
        doc.add_paragraph(
            f"Labelled F/M: {lf}/{lm}; Unlabelled F/M: {uf}/{um}; "
            f"{sex_test}, p={fmt_p(p_sex)}."
        )
    else:
        doc.add_paragraph("Sex data unavailable.")

    doc.add_heading("4. Readiness-warning audit",level=2)
    doc.add_paragraph(
        f"Verified warnings: Labelled {lw}/{len(labelled)}; "
        f"Unlabelled {uw}/{len(unlabelled)}; "
        f"{warn_test}, p={fmt_p(p_warn)}."
    )

    doc.add_heading("5. Slice-count audit",level=2)
    if slice_col is None:
        doc.add_paragraph(
            "No explicit analysed-slice-count variable was found. "
            "Do not substitute volume index or volume score."
        )
    else:
        doc.add_paragraph(f"Explicit slice-count column found: {slice_col}.")

    doc.add_heading("6. Reviewer issues",level=2)
    if issues:
        for issue in issues:
            doc.add_paragraph(issue, style="List Bullet")
    else:
        doc.add_paragraph("No critical issues detected.")

    doc.save(report_path)
    print("Saved:",report_path)
except Exception as e:
    print("Word report skipped:",e)


In [ ]:

# Save manifest
with open(OUTPUT_DIR/"Verification_Manifest.json","w",encoding="utf-8") as f:
    json.dump(verification_summary,f,ensure_ascii=False,indent=2)

print("="*70)
print("STEP 35A COMPLETE")
print("STATUS:",status)
print("OUTPUT:",OUTPUT_DIR)
print("="*70)



## Output files

```text
35A_Table3_Verifier_Output
├── Age_Distribution.png
├── Age_Boxplot.png
├── Table3_Verification_Audit.xlsx
├── Table3_Verification_Report.docx
└── Verification_Manifest.json
```

หลัง Run ให้ตรวจ 3 อย่างก่อน:
1. `STATUS`
2. Sheet `Warning Values`
3. Sheet `Recomputed Table3`

ถ้า warning ยังคงสูงผิดปกติ ให้ใช้ `Warning Values` หา field ที่ทำให้เกิด inflation ก่อนนำ Table 3 ลง manuscript
